In [21]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch.optim as optim

from pointNet import PointNet
import os
import glob
import numpy as np


Loading the data:

In [22]:
def get_scene_paths(dataset_path):
    """
    Get every path of each scene in the dataset
    """
    scene_dirs = []
    for area_dir in glob.glob(os.path.join(dataset_path, "Area*")):
        if os.path.isdir(area_dir):
            for scene_dir in glob.glob(os.path.join(area_dir, '*')):
                scene_dirs.append(scene_dir)

    return scene_dirs

def load_scene_data(scene_dir):
    label_files = glob.glob(os.path.join(scene_dir, "*_labels.txt"))
    if not label_files:
        raise FileNotFoundError(f"No labels found for {scene_dir}")

    label_file = label_files[0]
    base_name = label_file.replace("_labels.txt", "")
    
    point_file = None
    # Looking for the extensions. the extention should be .pcd 
    for ext in [".pcd", ".PCD", ".ply", ".PLY", ".txt", ".pts", ".xyz"]:
        potential_file = base_name + ext
        if os.path.exists(potential_file):
            point_file = potential_file
            break
            
    if point_file is None:
        raise FileNotFoundError(f"Point cloud error for base path: {base_name}")

    if point_file.lower().endswith(".pcd"):
        with open(point_file, 'rb') as f:
            header = []
            data_type = 'ascii'
            num_pts = 0
            fields = []
            sizes = []
            types = []
            
            # Skip the header
            while True:
                line_bytes = f.readline()
                if not line_bytes:
                    break
                line = line_bytes.decode('latin-1')
                header.append(line)
                if line.startswith("DATA"):
                    parts = line.strip().split()
                    if len(parts) > 1:
                        data_type = parts[1]
                    break
            
            for line in header:
                if line.startswith("POINTS"):
                    num_pts = int(line.strip().split()[1])
                elif line.startswith("WIDTH") and num_pts == 0:
                    num_pts = int(line.strip().split()[1])
                elif line.startswith("FIELDS"):
                    fields = line.strip().split()[1:]
                elif line.startswith("SIZE"):
                    sizes = [int(s) for s in line.strip().split()[1:]]
                elif line.startswith("TYPE"):
                    types = line.strip().split()[1:]

            # Loading the data:
            if data_type.lower() == 'ascii':
                points_data = np.loadtxt(f, encoding="latin-1")
                points = points_data[:, :3]
            else:
                binary_data = f.read()
                try:
                    dt_list = []
                    for field, size, t in zip(fields, sizes, types):
                        if t == 'F':
                            np_t = np.float32 if size == 4 else np.float64
                        elif t in ['I', 'U']:
                            np_t = np.int32 if size == 4 else (np.int16 if size == 2 else np.int8)
                        else:
                            np_t = np.float32
                        dt_list.append((field, np_t))
                    
                    dt = np.dtype(dt_list)
                    data = np.frombuffer(binary_data, dtype=dt, count=num_pts)
                    points = np.stack([data['x'], data['y'], data['z']], axis=-1)
                except Exception:
                    raw_data = np.frombuffer(binary_data, dtype=np.float32)
                    stride = len(raw_data) // num_pts
                    points = raw_data.reshape(num_pts, stride)[:, :3]
    else:
        points_data = np.loadtxt(point_file, encoding="latin-1")
        points = points_data[:, :3]

    # Loading the labels
    labels = np.loadtxt(label_file, dtype=np.int64, encoding="latin-1")
    
    assert len(points) == len(labels), f"[-] Error, number of points mismatch: {len(points)} pts vs {len(labels)} labels."

    return points, labels

Normalisation:

In [ ]:
def normalize_point_cloud(points):
    centroid = np.mean(points, axis=0)
    points_centered = points - centroid

    max_dist = np.max(np.sqrt(np.sum(points_centered ** 2, axis=1)))

    if max_dist > 0:
        points_normalized = points_centered / max_dist
    else:
        points_normalized = points_centered

    return points_normalized

In [ ]:
class Segmentation3dDataset(Dataset):
    def __init__(self, dataset_path, num_points = 2048, split="train"):
        super().__init__()
        self.num_points = num_points
        all_scenes = get_scene_paths(dataset_path=dataset_path)

        # Split Area 1 -> 5 : train, 
        # Area 6: val
        if split == "train":
            self.scene_dirs = [s for s in all_scenes if "Area_6" not in s]
        else:
            self.scene_dirs = [s for s in all_scenes if "Area_6" in s]

    def __len__(self):
        return len(self.scene_dirs)

    def __getitem__(self, idx):
        scene_dir = self.scene_dirs[idx]
        points, labels = load_scene_data(scene_dir= scene_dir)

        totals_pts = points.shape[0] 
        if totals_pts >= self.num_points:
            # Block_sampling:
            center_idx = np.random.choice(points.shape[0])
            center = points[center_idx]

            radius = 1.5
            distances = np.sqrt(np.sum((points - center) ** 2, axis = 1))

            mask = distances < radius
            block_points = points[mask]
            block_labels = labels[mask]

            total_pts = block_points.shape[0]
            if total_pts >= self.num_points:
                choice = np.random.choice(total_pts, self.num_points, replace=False)
            else:
                choice = np.random.choice(total_pts, self.num_points, replace=True)

            points_sampled = block_points[choice]
            labels_sampled = block_labels[choice]

        else:
            points_sampled = points
            labels_sampled = labels
                    
        points_normalized =  normalize_point_cloud(points_sampled)
        points_tensor = torch.tensor(points_normalized, dtype=torch.float32).transpose(1, 0)
        labels_tensor = torch.tensor(labels_sampled, dtype=torch.long)

        return points_tensor, labels_tensor
           

Load Model & Weights:

In [ ]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
DATASET_PATH = "C:/Projet/MachineLearning/PointCloudsSegmentation/test_export"
NUM_CLASSES = 13  
BATCH_SIZE = 16
NUM_EPOCHS = 250
LEARNING_RATE = 0.0005

train_dataset = Segmentation3dDataset(dataset_path=DATASET_PATH, num_points=2048, split="train")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)

model = PointNet(input_dims=3, output_dims=NUM_CLASSES).to(DEVICE)
state_dict = torch.load("pointnet_segmentation_s3dis.pt", weights_only=True)
model.load_state_dict(state_dict=state_dict)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999))
criterion = nn.CrossEntropyLoss()

model.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    
    for batch_idx, (points, labels) in enumerate(train_loader):
        # Push the data to GPU
        points = points.to(DEVICE)  # Shape : (B, 3, N)
        labels = labels.to(DEVICE)  # Shape : (B, N)
        
        # Reset the gradients:
        optimizer.zero_grad()
        
        outputs = model(points) 
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"---> End of epoch {epoch+1} | Avg Loss: {avg_loss:.4f}\n")

print("Training done!")

In [ ]:
checkpoint = {
    'epoch': NUM_EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}

save_path = "pointnet_segmentation_s3dis.pt"
torch.save(checkpoint, save_path)
print(f"Model saved at: {save_path}")

In [ ]:
%pip install plotly

Evaluation:

In [ ]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

model.eval()

val_dataset = Segmentation3dDataset(dataset_path=DATASET_PATH, num_points=1024000, split="val")
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)

points_tensor, labels_tensor = next(iter(val_loader))

with torch.no_grad():
    points_gpu = points_tensor.to(DEVICE)
    outputs = model(points_gpu)  # (1, num_classes, N)
    preds = torch.argmax(outputs, dim=1).squeeze(0).cpu().numpy()  # (N,)

points_np = points_tensor.squeeze(0).transpose(1, 0).numpy()  # (N, 3)
ground_truth = labels_tensor.squeeze(0).numpy()                # (N,)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
    subplot_titles=("<b>Vérité Terrain (Ground Truth)</b>", "<b>Prédictions PointNet</b>")
)

# 1. Ground Truth
fig.add_trace(
    go.Scatter3d(
        x=points_np[:, 0], y=points_np[:, 1], z=points_np[:, 2],
        mode='markers',
        marker=dict(
            size=3,
            color=ground_truth,
            colorscale='Viridis', 
            opacity=0.9,
            colorbar=dict(title="Classes GT", x=0.45)
        ),
        name='Ground Truth'
    ),
    row=1, col=1
)

# 2. Point Cloud
fig.add_trace(
    go.Scatter3d(
        x=points_np[:, 0], y=points_np[:, 1], z=points_np[:, 2],
        mode='markers',
        marker=dict(
            size=3,
            color=preds,
            colorscale='Viridis',
            opacity=0.9,
            colorbar=dict(title="Classes Pred", x=1.0)
        ),
        name='Prédictions'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="<b>S3DIS PointNet</b>",
    width=1200,
    height=700,
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data' 
    ),
    scene2=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data'
    )
)

fig.show()